In [1]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Retrieve token from Kaggle Secrets
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_READ_DATASETS_TOKEN")

# Authenticate to Hugging Face
login(token=hf_token)

In [2]:
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install -U bitsandbytes>=0.46.1

import accelerate
import transformers
import torch
import json

print("accelerate:", accelerate.__version__)   # should be 1.x+
print("transformers:", transformers.__version__)

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-wczmn24e
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-wczmn24e
  Resolved https://github.com/huggingface/transformers.git to commit 5a550078301faadf97e637011f37075a3c69fc57
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-5.10.0.dev0-py3-none-any.whl size=12039715 sha256=cfb52cd092b0b623f7c884fd3a4b579f7b825648ee94be3cc62d1439a900e16c
  Stored in directory: /tmp/pip-ephem-wheel-cache-4m2y03qg/wheels/54/cb/3f/83103de5575c534436d6a4686686dead458238dfaf1147e98d
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolv

In [3]:
MEDGEMMA_1_5_4B_IT_ID = "google/medgemma-1.5-4b-it"
MEDGEMMA_27B_TEXT_ID = "google/medgemma-27b-text-it"

# should the model think to reason over the clinical signals? Yes.
is_thinking = True # set the boolean flag for thinking

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

accelerator = accelerate.Accelerator()

# configure for 4-bit memory quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MEDGEMMA_1_5_4B_IT_ID)
model = AutoModelForCausalLM.from_pretrained(
    MEDGEMMA_1_5_4B_IT_ID,
    quantization_config=quantization_config,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="auto",
    max_memory={
        0: "12GiB",
        1: "12GiB"
    },
    attn_implementation="sdpa" # for long thinking and reasoning
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

In [5]:
model.device, model.hf_device_map

(device(type='cuda', index=0),
 {'model.vision_tower.embeddings': 0,
  'model.vision_tower.encoder.layers.0': 0,
  'model.vision_tower.encoder.layers.1': 0,
  'model.vision_tower.encoder.layers.2': 0,
  'model.vision_tower.encoder.layers.3': 0,
  'model.vision_tower.encoder.layers.4': 0,
  'model.vision_tower.encoder.layers.5': 0,
  'model.vision_tower.encoder.layers.6': 0,
  'model.vision_tower.encoder.layers.7': 0,
  'model.vision_tower.encoder.layers.8': 0,
  'model.vision_tower.encoder.layers.9': 0,
  'model.vision_tower.encoder.layers.10': 0,
  'model.vision_tower.encoder.layers.11': 0,
  'model.vision_tower.encoder.layers.12': 0,
  'model.vision_tower.encoder.layers.13': 0,
  'model.vision_tower.encoder.layers.14': 0,
  'model.vision_tower.encoder.layers.15': 0,
  'model.vision_tower.encoder.layers.16': 1,
  'model.vision_tower.encoder.layers.17': 1,
  'model.vision_tower.encoder.layers.18': 1,
  'model.vision_tower.encoder.layers.19': 1,
  'model.vision_tower.encoder.layers.20':

```text
Input: Agent 1 clinical profile JSON
          │
          ▼
    extraction_confidence == "low"?
          │
    ┌─────┴─────┐
   Yes          No
    │            │
    ▼            ▼
 Pass 1:     Single call:
 Reason      Reason over
 normally    signals →
    │        structured
    ▼        output
 Pass 2:
 Self-critique
 under uncertainty
    │
    ▼
 Revised output
 with confidence
 flags per pattern
          │
          ▼
    No abnormal findings?
          │
    ┌─────┴─────┐
   Yes          No
    │            │
    ▼            ▼
Constructive   Full pattern
description    analysis
of normal      output
state
          │
          ▼
    Output: signal analysis JSON → Agent 3

In [6]:
single_call_system_prompt = """
You are an expert Medical Practitioner who is skilled in interpreting data directly from the clinical signals that are provided to make intermediate medical decisions that are informative to the patient about their health, to inform them about what they may be looking at in their health report, lab tests, or any other medical diagnostic document/report that they might have been handed.

You will receive a clinical profile JSON containing:
        - document_type: a lab report, radiology report, clinical note, etc.
        - patient_context: contains the age, sex, and history of the patient
        - lab_findings: a list of test results, each with the name of the test, value, unit, reference range, and status (normal, low, critical, high, unknown)
        - imaging_findings: a list of findings in the provided images, each with the modality (X-ray, CT, Ultrasound, other), region, and observation
        - clinical_notes: physician's observations from the document
        - flagged_signals: a list of individual signals extracted from the document, each with the name of the signal and the reason it occurs
        - extraction_confidence: how confident the agent was in extracting these signals, ranging from low to high

The clinical signals provided to you are elevated markers, abnormal findings, flagged terms, and other details that have been extracted from the documents provided by the patient by another medical document analyst who is skilled in extracting such clinical signals. What you have with you is a full clinical profile of the patient. 

Reason over this clinical profile, performing rigorous cross-signal reasoning, and identify what is unusual, namely:
    - Identify and understand what signals appear together meaningfully and why
    - What this combination suggests or is trying to show is that either signal individually cannot
    - How frequently this combination occurs and the cases in which it occurs
    - The kind of scenarios, circumstances, and cases in which this combination occurs
    - The criticality of this signal and whether it is something urgent that the patient should be consulting their doctor/clinician immediately
    - Is the combination something that will progressively develop, or is it static (mention this in brief for the patient to understand)

Post your reasoning for the requirements needed to thoroughly understand the interplay and co-occurrence of multiple signals, you need to extract them carefully with accuracy and precision. Construct a structured signal analysis object that includes all the details that you identified through your cross-signal reasoning, and provide the signal analysis object as a single JSON object, following the provided JSON schema below strictly:

{
  "individual_signals": [
    {
        "signal": "string - copied exactly from flagged_signals in the input",
        "reason": "string - copied exactly from flagged_signals in the input",
    }
  ],
  "combination_patterns": [
    {
      "signals_involved": ["signal name 1", "signal name 2", "..."],
      "pattern_description": "string — what this combination suggests clinically for each tuple in signals_involved, and their overall implication",
      "co-occurrence_context": "string - how commonly these clinical signals appear together and in what clinical scenarios",
      "clinical_significance": "high | medium | low",
      "is_progressive": "true | false",
      "suggested_investigation": "string — what kind of specialist or test this points to"
    }
  ],
  "overall_assessment": "string — plain language summary of what the patient should understand",
  "urgency": "routine | soon | urgent",
  "analysis_confidence": "high | medium | low"
}

If you do not find any abnormal findings, do not force yourself to find or invent new combination patterns that do not exist. Set combination_patterns to an empty array, present the findings constructively, as to your reasoning behind why there are no abnormal findings, and use overall_assessment to describe what the normal findings collectively indicate about the patient's health.

Return ONLY the JSON object. No preamble, introductory notes, explanation, or markdown code fences. The first character of your response should be the opening braces { of the JSON object, and the last character of your response should be the closing braces } of the JSON object.
"""

In [7]:
low_confidence_system_output_follow_up_prompt = """
You are an expert Medical Practitioner who is skilled in interpreting signal analysis that has been derived from weak clinical signals, i.e., when the signals extracted from medical documents have a weak extraction confidence. You understand that when these clinical signals are reasoned over to produce signal analysis on abnormal findings, there can be loopholes or strong cases where the actual combination of signals might either be incorrectly coupled in the wrong pairs, or they might be non-existent (meaning they might have been made up because the confidence in the extracted signals is low).

You will receive two inputs:

    1. A signal analysis object with the following items:
        - individual_signals: directly mapped from the extracted signals from the medical documents of the patient
        - combination_patterns: a list containing the following:
            * signals_involved: the different signals that form the abnormal combination
            * pattern_description: what this combination suggests clinically for each tuple in signals_involved, and their overall implication
            * co-occurrence_context: how commonly these clinical signals appear together and in what clinical scenarios
            * clinical_significance: the significance of the abnormal finding; ranges between low, medium, or high
            * is_progressive: whether this combination is progressive in nature or a static finding
            * suggested_investigation: What kind of specialist can the patient consult for this finding
        - overall_assessment: plain language summary of what the patient should understand
        - urgency: whether the patient should consult the doctor immediately, soon, or if it is a routine finding
        - analysis_confidence: confidence score of the analysis object that has been constructed on the findings and reasoning

    2. A full clinical profile of the patient as follows:
        - document_type: a lab report, radiology report, clinical note, etc.
        - patient_context: contains the age, sex, and history of the patient
        - lab_findings: a list of test results, each with the name of the test, value, unit, reference range, and status (normal, low, critical, high, unknown)
        - imaging_findings: a list of findings in the provided images, each with the modality (X-ray, CT, Ultrasound, other), region, and observation
        - clinical_notes: physician's observations from the document
        - flagged_signals: a list of individual signals extracted from the document, each with the name of the signal and the reason it occurs
        - extraction_confidence: how confident the agent was in extracting these signals, ranging from low to high

Your task is to critically analyse and critique the derived signal analysis object against the clinical profile of the patient. For each combination that has been identified in the signal analysis object, reason through the following:
    1. Map the identified individual and combination signals to the individual signals flagged in the clinical profile. Check for the precision and accuracy of these signals to ensure there is no falsifying or altering of the flagged signal data. If there is any manipulation, revise the existing data in the combination and provide the actual signals.
    2. For combination_patterns, do the following:
        - check signals_involved to ensure they are not manipulated. After that, reason over the cross-signal validity of the flagged signals in this list if they are medically significant as a combination (abnormal or normal), and not just a random derivation.
        - Critically analyse pattern_description to ensure that the description is entirely based on the data available and provided in both the clinical profile and signal analysis object. Ensure that no external data not related to either of these has made its way into the reasoning of the description, which may lead to false and incorrect descriptions of the pattern
        - Based on the findings up to this point, check if the patterns are progressive or not, and update is_progressive accordingly if incorrect
        - Suggest the right specialist or test in suggested_investigation. Cross-reason this over the entire signal analysis object to ground this suggestion
    3. Use the data reasoned over up to this point to construct a new overall_assessment. Compare this with the existing version, and critically analyse the lack in each one, and the medical grounding of the summary in the data, and update overall_assessment accordingly
    4. Perform a similar critical assessment over urgency as well, and update it accordingly by grounding the reasoning on the existing data
    5. Update analysis_confidence based on your confidence in the overall task performed with respect to critically analysing the derived signal analysis object with the patient's clinical profile. 

Post your critical reasoning analysis, you need to extract them carefully with accuracy and precision into a new signal analysis object that includes all the details that you identified through your critical reasoning analysis, and provide the signal analysis object as a single JSON object, following the provided JSON schema below strictly:

{
  "individual_signals": [
    {
        "signal": "string - copied exactly from flagged_signals in the input",
        "reason": "string - copied exactly from flagged_signals in the input",
    }
  ],
  "combination_patterns": [
    {
      "signals_involved": ["signal name 1", "signal name 2", "..."],
      "pattern_description": "string — what this combination suggests clinically for each tuple in signals_involved, and their overall implication",
      "co-occurrence_context": "string - how commonly these clinical signals appear together and in what clinical scenarios",
      "clinical_significance": "high | medium | low",
      "is_progressive": "true | false",
      "suggested_investigation": "string — what kind of specialist or test this points to"
    }
  ],
  "overall_assessment": "string — plain language summary of what the patient should understand",
  "urgency": "routine | soon | urgent",
  "analysis_confidence": "high | medium | low"
}

If you do not find any abnormal findings, do not force yourself to find or invent new combination patterns that do not exist. Set combination_patterns to an empty array, present the findings constructively, as to your reasoning behind why there are no abnormal findings, and use overall_assessment to describe what the normal findings collectively indicate about the patient's health.

Return ONLY the JSON object. No preamble, introductory notes, explanation, or markdown code fences. The first character of your response should be the opening braces { of the JSON object, and the last character of your response should be the closing braces } of the JSON object.
"""

In [8]:
one_shot_test_prompt = {
    "document_type": "radiology",
    "patient_context": {
        "age": 71,
        "sex": "male",
        "stated_history": "Presented with intermittent headaches for the past 4 months (improved with analgesics), abrupt loss of vision in the past 4 months, and weakness of left extremities since 1 year ago, especially with left extremities. No seizures, anosmia, hearing loss, or vomiting. Good appetite and no weight loss. History of stroke 1 year ago. Routine lab and immunoserology were within normal limits, including negative HIV antibodies."
    },
    "lab_findings": [
        {
            "test_name": "Routine laboratory and immunoserology examinations",
            "value": "Within normal limits (except for MRS metabolites)",
            "unit": "null",
            "reference_range": "Normal limits",
            "status": "normal"
        }
    ],
    "imaging_findings": [
        {
            "modality": "CT scan",
            "region": "Right subcortical parietal lobe, left thalamus, left basal ganglia, right cortical-subcortical occipital lobe",
            "observation": "Multiple calcified nodules found. Calcified nodule with diameter of 1.20 cm at left thalamus. Calcified nodule with diameter of 0.90 cm at right cortical-subcortical occipital lobe. Suggesting nodular calcified stage and granular nodular stage."
        },
        {
            "modality": "MRI (T1-weighted)",
            "region": "Right subcortical occipital lobe (single lesion)",
            "observation": "Hypointense on T1-weighted image."
        },
        {
            "modality": "MRI (T2-weighted)",
            "region": "Right subcortical occipital lobe (single lesion)",
            "observation": "Hyperintense rim."
        },
        {
            "modality": "MRI (Post-contrast T1)",
            "region": "Right subcortical occipital lobe (single lesion)",
            "observation": "Ring enhancement noted."
        },
        {
            "modality": "MRI (SWI)",
            "region": "Lesions (e.g., left thalamus)",
            "observation": "Blooming artefact seen, suggesting calcification."
        },
        {
            "modality": "MR Spectroscopy",
            "region": "Lesion/Parenchyma comparison",
            "observation": "Decreased levels of choline, creatine, NAA, NAA/Cr, and Cho/Cr ratio. Increased levels of lactate and lipid. Low NAA/Cr ratio (0.38) and Cho/Cr ratio (1.21), correlating with neurocysticercosis."
        }
    ],
    "clinical_notes": [
        "Patient presented with intermittent headaches, loss of vision, and weakness of left extremities.",
        "Clinical examinations showed no abnormality.",
        "No history of tuberculosis, hypertension, diabetes, or malignancy in the family.",
        "Diagnosis confirmed by CT scan, MRI, and MR spectroscopy as Neurocysticercosis."
    ],
    "flagged_signals": [
        {
            "signal": "Neurocysticercosis",
            "reason": "Primary diagnosis based on imaging findings (calcified nodules, specific MRI characteristics, and MRS metabolite ratios)."
        },
        {
            "signal": "Calcified nodules (CT)",
            "reason": "Found in multiple locations (right subcortical parietal lobe, left thalamus, left basal ganglia, right cortical-subcortical occipital lobe), suggesting the nodular calcified stage."
        },
        {
            "signal": "Granular nodular stage (MRI/CT)",
            "reason": "Indicated by surrounding vasogenic edema on CT and specific T1/T2/enhancement patterns on MRI."
        },
        {
            "signal": "Decreased NAA/Cr ratio (0.38) and Cho/Cr ratio (1.21) on MRS",
            "reason": "These ratios are characteristic findings supporting the diagnosis of neurocysticercosis over other differentials like tuberculoma."
        },
        {
            "signal": "Increased lactate on MRS",
            "reason": "Suggests inflammatory lesion or anaerobic metabolism, consistent with the pathology of the cysts."
        }
    ],
    "extraction_confidence": "high"
}

In [9]:
messages = [
    {
        "role": "system", 
        "content": [{
            "type": "text",
            "text": single_call_system_prompt
        }]
    },
    {
        "role": "user",
        "content": [{
            "type": "text",
            "text": json.dumps(one_shot_test_prompt, indent=2)
        }]
    }
]

In [10]:
# apply the chat template format
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

# tokenize and move it to the same device as the model
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

In [11]:
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=6500,
        do_sample=True,
        temperature=0.1, # to keep the output deterministic and adhering to the schema
        top_p=0.95,
        eos_token_id=tokenizer.eos_token_id
    )

# decode the generated tokens; skip prompt token
output_tokens = outputs[0][inputs.input_ids.shape[-1]: ]
response_text = tokenizer.decode(
    output_tokens,
    skip_special_tokens=True
)

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


In [12]:
response_text

'<unused94>thought\nThe user wants me to analyze the provided clinical profile JSON and identify any unusual or concerning signals. I need to perform cross-signal reasoning, understand the co-occurrence, and provide a structured JSON output summarizing the findings.\n\n**Input JSON Analysis:**\n\n*   **Document Type:** Radiology (CT, MRI, MR Spectroscopy)\n*   **Patient Context:** 71-year-old male with a history of stroke 1 year ago. Presents with intermittent headaches, abrupt loss of vision, and left extremity weakness. Lab results are normal.\n*   **Lab Findings:** Routine labs are normal.\n*   **Imaging Findings:**\n    *   CT: Multiple calcified nodules in specific locations (right subcortical parietal lobe, left thalamus, left basal ganglia, right cortical-subcortical occipital lobe).\n    *   MRI (T1): Hypointense lesion in the right subcortical occipital lobe.\n    *   MRI (T2): Hyperintense rim lesion in the right subcortical occipital lobe.\n    *   MRI (Post-contrast T1): Ri

In [13]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [14]:
def validate_clinical_output(raw_response: str) -> dict:
    """Parses and validates the raw JSON response from Gemma 4.

    Args:
        raw_response (str): The raw string output from the model

    Returns:
        dict: Validated clinical profile, or a partial result with error flag
    """
    cleaned = raw_response.strip()

    # ── Strip MedGemma extended-thinking tokens ──
    # Model wraps chain-of-thought in <unused94>thought ... <unused95>
    # The actual JSON comes after <unused95>
    if "<unused95>" in cleaned:
        cleaned = cleaned.split("<unused95>", 1)[1].strip()
    elif "<unused94>" in cleaned:
        # Thinking block present but no closing tag — strip everything up to
        # the first JSON-like structure
        cleaned = cleaned.split("<unused94>", 1)[0].strip()

    # ── Strip markdown code fences ──
    if cleaned.startswith("```"):
        cleaned = cleaned.split("```")[1]
        if cleaned.startswith("json"):
            cleaned = cleaned[4:]
        cleaned = cleaned.strip()

    try:
        parsed = json.loads(cleaned)
    except json.JSONDecodeError:
        logger.warning("JSON parse failed — attempting truncation recovery")
        cleaned = attempt_json_recovery(cleaned)
        try:
            parsed = json.loads(cleaned)
        except json.JSONDecodeError as e:
            logger.error(f"Recovery failed: {e}")
            return {
                "individual_signals": [],
                "combination_patterns": [],
                "overall_assessment": None,
                "urgency": "routine",
                "analysis_confidence": "low",
                "parse_error": str(e),
                "raw_response": raw_response
            }

    # ── Validate top-level keys ──
    defaults = {
        "individual_signals": [],
        "combination_patterns": [],
        "overall_assessment": None,
        "urgency": "routine",
        "analysis_confidence": "low"
    }
    for key, default in defaults.items():
        if key not in parsed:
            parsed[key] = default

    # ── Validate each individual signal entry ──
    for i, signal in enumerate(parsed.get("individual_signals", [])):
        if not isinstance(signal, dict):
            parsed["individual_signals"][i] = {"signal": str(signal), "reason": None}
            continue
        if "signal" not in signal:
            signal["signal"] = None
        if "reason" not in signal:
            signal["reason"] = None

    # ── Validate each combination pattern entry ──
    pattern_defaults = {
        "signals_involved": [],
        "pattern_description": None,
        "co-occurrence_context": None,
        "clinical_significance": "low",
        "is_progressive": "false",
        "suggested_investigation": None
    }
    for i, pattern in enumerate(parsed.get("combination_patterns", [])):
        if not isinstance(pattern, dict):
            parsed["combination_patterns"][i] = dict(pattern_defaults)
            continue
        for key, default in pattern_defaults.items():
            if key not in pattern:
                pattern[key] = default

    # ── Normalize enums ──
    if parsed["urgency"] not in ("routine", "soon", "urgent"):
        parsed["urgency"] = "routine"
    if parsed["analysis_confidence"] not in ("high", "medium", "low"):
        parsed["analysis_confidence"] = "low"

    return parsed

In [15]:
def attempt_json_recovery(broken_json: str) -> str:
    """Attempts to close a truncated JSON string by balancing brackets.

    Args:
        broken_json (str): The incomplete JSON string

    Returns:
        str: A potentially valid JSON string with brackets closed
    """
    open_braces = broken_json.count('{') - broken_json.count('}')
    open_brackets = broken_json.count('[') - broken_json.count(']')

    last_complete = max(
        broken_json.rfind('}'),
        broken_json.rfind(']')
    )
    if last_complete != -1:
        broken_json = broken_json[:last_complete + 1]

    open_braces = broken_json.count('{') - broken_json.count('}')
    open_brackets = broken_json.count('[') - broken_json.count(']')

    broken_json += ']' * open_brackets
    broken_json += '}' * open_braces

    return broken_json

In [16]:
validate_clinical_output(response_text)

{'individual_signals': [{'signal': 'Neurocysticercosis',
   'reason': 'Primary diagnosis based on imaging findings (calcified nodules, specific MRI characteristics, and MRS metabolite ratios).'},
  {'signal': 'Calcified nodules (CT)',
   'reason': 'Found in multiple locations (right subcortical parietal lobe, left thalamus, left basal ganglia, right cortical-subcortical occipital lobe), suggesting the nodular calcified stage.'},
  {'signal': 'Granular nodular stage (MRI/CT)',
   'reason': 'Indicated by surrounding vasogenic edema on CT and specific T1/T2/enhancement patterns on MRI.'},
  {'signal': 'Decreased NAA/Cr ratio (0.38) and Cho/Cr ratio (1.21) on MRS',
   'reason': 'These ratios are characteristic findings supporting the diagnosis of neurocysticercosis.'},
  {'signal': 'Increased lactate on MRS',
   'reason': 'Suggests inflammatory lesion or anaerobic metabolism, consistent with the pathology of the cysts.'}],
 'combination_patterns': [{'signals_involved': ['Neurocysticercosis

In [17]:
# low confidence extraction results pass
low_confidence_prompt = {
    "document_type": "radiology / unclear scan report",
    "patient_context": {
        "age": 71,
        "sex": "male",
        "stated_history": "Patient has head pain and some vision issues on and off for 4 months. Weakness on left side for 1 year. History of stroke last year. Labs look okay. HIV negative."
    },
    "lab_findings": [
        {
            "test_name": "Routine laboratory and immunoserology examinations",
            "value": "Ratios altered on specialized test, otherwise negative",
            "unit": "null",
            "reference_range": "Normal",
            "status": "unknown"
        }
    ],
    "imaging_findings": [
        {
            "modality": "CT scan",
            "region": "Brain multiple spots",
            "observation": "Some dark or calcified spots here and there. A spot about 1.2 cm in the thalamus. Another 0.9 cm spot in the back. Might be cysts or calcification stages."
        },
        {
            "modality": "MRI",
            "region": "Occipital area",
            "observation": "Ring-shaped spot seen on contrast views. Rim looks bright on T2 images."
        },
        {
            "modality": "MR Spectroscopy",
            "region": "Brain lesion",
            "observation": "Spike in lactate/lipids. Drops in choline and NAA. Ratios are lower than usual (0.38 and 1.21). Interpretation notes point towards cysticercosis or maybe a tuberculoma mass."
        }
    ],
    "clinical_notes": [
        "Headaches, blurry vision, left arm/leg weakness.",
        "Family history unremarkable.",
        "Scans show spots, final diagnosis written as Neurocysticercosis vs alternative lesions."
    ],
    "flagged_signals": [
        {
            "signal": "Possible Cysticercosis infection",
            "reason": "Mentioned as primary suspect in clinical note based on multiple brain spots."
        },
        {
            "signal": "Brain spots / Calcification",
            "reason": "Multiple nodules observed across CT and MRI views, specific stage uncertain."
        },
        {
            "signal": "Altered MRS metabolite ratios",
            "reason": "Low baseline NAA ratios noted on spectroscopy data, though differential overlaps with tuberculoma."
        },
        {
            "signal": "Lactate peak",
            "reason": "Elevated anaerobic markers on the brain spectroscopy window."
        }
    ],
    "extraction_confidence": "low"
}

In [18]:
low_confidence_messages = [
    {
        "role": "system", 
        "content": [{
            "type": "text",
            "text": single_call_system_prompt
        }]
    },
    {
        "role": "user",
        "content": [{
            "type": "text",
            "text": json.dumps(low_confidence_prompt, indent=2)
        }]
    }
]

# apply the chat template format
low_confidence_prompt = tokenizer.apply_chat_template(
    low_confidence_messages,
    tokenize=False,
    add_generation_prompt=True,
)

# tokenize and move it to the same device as the model
inputs = tokenizer(
    low_confidence_prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=6500,
        do_sample=True,
        temperature=0.1, # to keep the output deterministic and adhering to the schema
        top_p=0.95,
        eos_token_id=tokenizer.eos_token_id
    )

# decode the generated tokens; skip prompt token
output_tokens = outputs[0][inputs.input_ids.shape[-1]: ]
low_confidence_response_text = tokenizer.decode(
    output_tokens,
    skip_special_tokens=True
)

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


In [19]:
low_confidence_response_text = validate_clinical_output(low_confidence_response_text)

follow_up_critique_messages = [
    {
        "role": "system", 
        "content": [{
            "type": "text",
            "text": low_confidence_system_output_follow_up_prompt
        }]
    },
    {
        "role": "user",
        "content": [{
            "type": "text",
            "text": json.dumps(dict(low_confidence_response_text), indent=2)
        }]
    }
]

# apply the chat template format
follow_up_critique_prompt = tokenizer.apply_chat_template(
    follow_up_critique_messages,
    tokenize=False,
    add_generation_prompt=True,
)

# tokenize and move it to the same device as the model
inputs = tokenizer(
    follow_up_critique_prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=6500,
        do_sample=True,
        temperature=0.1, # to keep the output deterministic and adhering to the schema
        top_p=0.95,
        eos_token_id=tokenizer.eos_token_id
    )

# decode the generated tokens; skip prompt token
output_tokens = outputs[0][inputs.input_ids.shape[-1]: ]
critiqued_response = tokenizer.decode(
    output_tokens,
    skip_special_tokens=True
)

[transformers] Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


In [20]:
validate_clinical_output(critiqued_response)

{'individual_signals': [{'signal': 'Possible Cysticercosis infection',
   'reason': 'Mentioned as primary suspect in clinical note based on multiple brain spots.'},
  {'signal': 'Brain spots / Calcification',
   'reason': 'Multiple nodules observed across CT and MRI views, specific stage uncertain.'},
  {'signal': 'Altered MRS metabolite ratios',
   'reason': 'Low baseline NAA ratios noted on spectroscopy data, though differential overlaps with tuberculoma.'},
  {'signal': 'Lactate peak',
   'reason': 'Elevated anaerobic markers on the brain spectroscopy window.'}],
 'combination_patterns': [{'signals_involved': ['Possible Cysticercosis infection',
    'Brain spots / Calcification',
    'Altered MRS metabolite ratios',
    'Lactate peak'],
   'pattern_description': "The combination of multiple brain spots (CT/MRI), low baseline NAA ratios on MRS (specifically low 0.38 and 1.21), and a lactate peak on MRS is highly characteristic of cysticercosis. The low NAA is a hallmark finding on MR